# 00 — Prepare data

Builds the four datasets, each stored **alone** with readable column names.

| dataset | modality | context |
|---|---|---|
| `diabetes_130` | tabular | health |
| `framingham` | tabular | health |
| `german_credit` | tabular | loan_finance |
| `civilcomments` | text | general |

Columns are kept interpretable on purpose: the physical-safety
specifications in `dataset_specs.py` (plausible value ranges,
safety-critical strata, protected attributes) refer to them by name.
One-hot encoding happens at model time, inside `safety_lib.numeric_design`.

**Units.** Everything downstream is a fraction in `[0, 1]`. No percents.

Run order: **00 → 01 → 02 → 02b (GPU) → 03 → 04**

In [1]:
# --- Colab setup -----------------------------------------------------------
!pip install numpy pandas
# Upload safety_lib.py and dataset_specs.py next to this notebook first.
!pip install -q ucimlrepo pandas numpy scikit-learn xgboost matplotlib
# For the text arm as well:
!pip install -q datasets detoxify transformers torch

import os
import sys

sys.path.insert(0, os.getcwd())

import json
import numpy as np
import pandas as pd

import safety_lib as sl
from dataset_specs import ALL_SPECS, DIABETES, FRAMINGHAM, GERMAN_CREDIT, CIVILCOMMENTS, spec_frame

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)
print(f"safety_lib {sl.VERSION} | units = {sl.UNITS} | data -> {sl.DATA}")
print(spec_frame().to_string(index=False))

  Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2026.4-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl (12.7 MB)
Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl (10.0 MB)
Using cached tzdata-2026.4-py2.py3-none-any.whl (347 kB)

   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


safety_lib 2.0-notebooks | units = fraction | data -> D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data
      dataset modality      context  label_column text_column sensitive_columns  n_value_ranges  n_edge_case_strata  min_stratum_n                                                                                                                           path
 diabetes_130  tabular       health readmit_early         NaN       race|gender               8                   5            500       D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\diabetes_130.csv
   framingham  tabular       health    TenYearCHD         NaN     sex|age_group               8                   5            120         D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\framingham.csv
german_credit  tabular loan_finance    bad_credit         NaN     se

## Which parts to build

Set a flag to `False` to skip a dataset (for example if you already built it,
or if you are only working the tabular arm today).

In [2]:
BUILD = {
    "diabetes_130": True,
    "framingham": True,
    "german_credit": True,
    "civilcomments": True,
}

# CivilComments pool sizes. The text arm mixes 10,000 records per dose, so the
# clean pool must comfortably exceed that and the harmful pools must cover the
# largest dose (0.40 x 10,000 = 4,000 records).
N_CLEAN, N_TOXIC, N_THREAT, N_PROMPTS = 40_000, 20_000, 8_000, 1_000
CLEAN_MAX, TOXIC_MIN, THREAT_MIN = 0.10, 0.50, 0.50     # human annotation cut-offs

## Diabetes 130 (UCI id=296) — health

`age` arrives as a bucket string (`[70-80)`); we add a numeric `age_years`
midpoint so the safety-critical strata queries can use it. Nothing is
dropped, one-hot encoded or subsampled here.

In [5]:
def build_diabetes():
    from ucimlrepo import fetch_ucirepo

    ds = fetch_ucirepo(id=296)
    X = ds.data.features.copy()
    y = (ds.data.targets.iloc[:, 0].values == "<30").astype(int)

    X = X.replace("?", np.nan)
    X = X.drop(columns=[c for c in ("weight", "payer_code", "medical_specialty")
                        if c in X.columns])

    # numeric midpoint for the age bucket, used by the edge-case strata
    def age_mid(a):
        try:
            lo, hi = str(a).strip("[)").split("-")
            return (float(lo) + float(hi)) / 2.0
        except Exception:
            return np.nan

    X["age_years"] = X["age"].map(age_mid)
    X["race"] = X["race"].fillna("Unknown")

    out = X.copy()
    out[DIABETES.label_column] = y
    out.to_csv(DIABETES.path, index=False)
    print(f"[wrote] {DIABETES.path}  {out.shape}")
    print(f"        early-readmission base rate = {y.mean():.4f}")
    print(f"        race groups: {out['race'].value_counts().to_dict()}")
    return out

In [6]:
FRAMINGHAM_MIRRORS = [
    "https://raw.githubusercontent.com/GauravPadawe/Framingham-Heart-Study/master/framingham.csv",
    "https://raw.githubusercontent.com/vsnupoudel/Coursera-Data-Science/master/framingham.csv",
]


def build_framingham():
    raw = None
    local = sl.DATA / "framingham_raw.csv"
    if local.exists():
        raw = pd.read_csv(local)
        print(f"[read] {local} (uploaded copy)")
    else:
        for url in FRAMINGHAM_MIRRORS:
            try:
                raw = pd.read_csv(url)
                print(f"[read] {url}")
                break
            except Exception as e:
                print(f"[miss] {url}: {type(e).__name__}")
    if raw is None:
        raise FileNotFoundError(
            "Framingham not reachable. Upload framingham.csv to "
            f"{sl.DATA / 'framingham_raw.csv'} and re-run this cell.")

    df = raw.copy()
    df.columns = [c.strip() for c in df.columns]
    if "male" in df.columns:
        df["sex"] = np.where(df["male"] == 1, "male", "female")
    df["age_group"] = pd.cut(df["age"], [0, 45, 55, 65, 200],
                             labels=["<45", "45-54", "55-64", "65+"]).astype(str)
    df = df.dropna(subset=[FRAMINGHAM.label_column])
    df[FRAMINGHAM.label_column] = df[FRAMINGHAM.label_column].astype(int)
    df.to_csv(FRAMINGHAM.path, index=False)
    print(f"[wrote] {FRAMINGHAM.path}  {df.shape}")
    print(f"        10-year CHD base rate = {df[FRAMINGHAM.label_column].mean():.4f}")
    print(f"        sex counts: {df['sex'].value_counts().to_dict()}")
    return df

## Statlog German Credit (UCI id=144) — loan_finance

Attribute codes are renamed to the documented names; `sex` is derived from
the combined personal-status/sex attribute, which is exactly the kind of
protected-attribute entanglement the representation sub-dimension is for.

In [7]:
GERMAN_COLS = [
    "checking_status", "duration_months", "credit_history", "purpose",
    "credit_amount", "savings_status", "employment_since", "installment_rate",
    "personal_status_sex", "other_debtors", "residence_since", "property",
    "age_years", "other_installment_plans", "housing", "existing_credits",
    "job", "num_dependents", "telephone", "foreign_worker",
]
SEX_MAP = {"A91": "male", "A92": "female", "A93": "male", "A94": "male", "A95": "female"}


def build_german():
    from ucimlrepo import fetch_ucirepo

    ds = fetch_ucirepo(id=144)
    X = ds.data.features.copy()
    if len(X.columns) == len(GERMAN_COLS):
        X.columns = GERMAN_COLS
    t = ds.data.targets.iloc[:, 0]
    y = (pd.to_numeric(t, errors="coerce") == 2).astype(int)   # 2 = bad risk

    X["sex"] = X["personal_status_sex"].map(SEX_MAP).fillna("unknown")
    X["age_group"] = pd.cut(X["age_years"], [0, 25, 35, 50, 200],
                            labels=["<=25", "26-35", "36-50", "51+"]).astype(str)
    X[GERMAN_CREDIT.label_column] = y.to_numpy()
    X.to_csv(GERMAN_CREDIT.path, index=False)
    print(f"[wrote] {GERMAN_CREDIT.path}  {X.shape}")
    print(f"        bad-credit base rate = {y.mean():.4f}")
    print(f"        sex counts: {X['sex'].value_counts().to_dict()}")
    return X

## CivilComments — general (text)

Three pools split by the **human** annotation, never by a classifier — that
is what keeps the later calibration check non-circular:

* `clean`  — `toxicity <= 0.10`
* `toxic`  — `toxicity >= 0.50`   (targets `harm_content_density`)
* `threat` — `threat  >= 0.50`   (targets `threat_density`)

Plus `realtoxicityprompts.jsonl`, the evaluation prompts used in 02b, and a
`civilcomments_base.csv` clean baseline dataset for notebook 01.

In [8]:
def build_civilcomments():
    from datasets import load_dataset

    cc = load_dataset("google/civil_comments", split="train").shuffle(seed=0)
    clean, toxic, threat = [], [], []
    for row in cc:
        t = (row.get("text") or "").strip()
        if len(t) < 20:
            continue
        tox = float(row.get("toxicity") or 0.0)
        thr = float(row.get("threat") or 0.0)
        if thr >= THREAT_MIN and len(threat) < N_THREAT:
            threat.append(t)
        elif tox <= CLEAN_MAX and len(clean) < N_CLEAN:
            clean.append(t)
        elif tox >= TOXIC_MIN and len(toxic) < N_TOXIC:
            toxic.append(t)
        if len(clean) >= N_CLEAN and len(toxic) >= N_TOXIC and len(threat) >= N_THREAT:
            break

    def dump(path, texts):
        with open(path, "w") as f:
            for t in texts:
                f.write(json.dumps({"text": t}) + "\n")
        print(f"[wrote] {path}  ({len(texts)} lines)")

    dump(CIVILCOMMENTS.clean_pool, clean)
    dump(CIVILCOMMENTS.toxic_pool, toxic)
    dump(CIVILCOMMENTS.threat_pool, threat)

    # the clean baseline dataset scored in notebook 01
    base = pd.DataFrame({CIVILCOMMENTS.text_column: clean[:CIVILCOMMENTS.mix_size]})
    base.to_csv(CIVILCOMMENTS.path, index=False)
    print(f"[wrote] {CIVILCOMMENTS.path}  {base.shape}")

    rtp = load_dataset("allenai/real-toxicity-prompts", split="train").shuffle(seed=0)
    prompts = []
    for row in rtp:
        p = ((row.get("prompt") or {}).get("text") or "").strip()
        if len(p) >= 20:
            prompts.append(p)
        if len(prompts) >= N_PROMPTS:
            break
    dump(sl.DATA / "realtoxicityprompts.jsonl", prompts)
    return base

In [9]:
BUILDERS = {
    "diabetes_130": build_diabetes,
    "framingham": build_framingham,
    "german_credit": build_german,
    "civilcomments": build_civilcomments,
}

manifest = []
for name, wanted in BUILD.items():
    if not wanted:
        print(f"[skip] {name} (BUILD flag off)")
        continue
    try:
        BUILDERS[name]()
        manifest.append({"dataset": name, "status": "built", "detail": ""})
    except Exception as e:
        print(f"[FAIL] {name}: {type(e).__name__}: {e}")
        manifest.append({"dataset": name, "status": "failed",
                         "detail": f"{type(e).__name__}: {e}"})

D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\.venv\Lib\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


[wrote] D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\diabetes_130.csv  (101766, 46)
        early-readmission base rate = 0.1116
        race groups: {'Caucasian': 76099, 'AfricanAmerican': 19210, 'Unknown': 2273, 'Hispanic': 2037, 'Other': 1506, 'Asian': 641}
[read] https://raw.githubusercontent.com/GauravPadawe/Framingham-Heart-Study/master/framingham.csv
[wrote] D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\framingham.csv  (4240, 18)
        10-year CHD base rate = 0.1519
        sex counts: {'female': 2420, 'male': 1820}
[wrote] D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\german_credit.csv  (1000, 23)
        bad-credit base rate = 0.3000
        sex counts: {'male': 690, 'female': 310}


D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 97320/97320 [00:00<00:00, 658218.61 examples/s]


[wrote] D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\civilcomments_clean.jsonl  (40000 lines)
[wrote] D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\civilcomments_toxic.jsonl  (20000 lines)
[wrote] D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\civilcomments_threat.jsonl  (4125 lines)
[wrote] D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\civilcomments_base.csv  (10000, 1)


D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joyce\.cache\huggingface\hub\datasets--allenai--real-toxicity-prompts. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but 

[wrote] D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\realtoxicityprompts.jsonl  (1000 lines)


## Manifest

Every notebook writes and prints its CSVs. This one records what got built.

In [10]:
man = pd.DataFrame(manifest)
for s in ALL_SPECS:
    hit = man["dataset"] == s.name
    if hit.any():
        p = sl.Path(s.path)
        man.loc[hit, "rows"] = (len(pd.read_csv(p)) if p.exists() else np.nan)
        man.loc[hit, "path"] = s.path
man["units"] = sl.UNITS
sl.write_csv(man, "00_data_manifest.csv")


[csv] D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\results\00_data_manifest.csv   (4 rows x 6 cols, units=fraction)
      dataset status detail     rows                                                                                                                           path    units
 diabetes_130  built        101766.0       D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\diabetes_130.csv fraction
   framingham  built          4240.0         D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\framingham.csv fraction
german_credit  built          1000.0      D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\data\german_credit.csv fraction
civilcomments  built         10000.0 D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\d

WindowsPath('D:/Documents/PhD_program/DISSERTATION_RELATED/Dissertation_Studies_code/safety_notebooks/notebooks/results/00_data_manifest.csv')